In [2]:
# =============================================================================
# FINE-TUNING WHISPER ON YOUR DOMAIN
# =============================================================================
#
# Course map:
#   Phase 04 → Transfer learning & fine-tuning (general)
#   Phase 06 → Whisper architecture & fine-tuning
#   Phase 11 → LoRA & QLoRA (train few extra weights, keep base frozen)
#
# Today's deliverable (target):
#   LoRA-tuned Whisper that hears SALES jargon better
#   (WER on terms like "SOC 2", "$299", "Professional" ~25% → <5%)
#   Drop-in replacement for the ASR step in your live earpiece loop.
#
# ---------------------------------------------------------------------------
# THE PROBLEM (easy picture)
# ---------------------------------------------------------------------------
# Stock Whisper is great on general English. On a sales call it still mishears:
#   "SOC 2" → "sock too"     "$299" → "two ninety nine" messy
#   "Professional plan" → "professional planned"
#
# Your earpiece Correction-GPT then starts from BAD text → worse coaching.
#
# Fine-tuning = show Whisper many (audio → correct transcript) examples from
# YOUR domain so those words become easy.
#
#   Generic Whisper          Domain-tuned Whisper
#   "sock too compliant"  →  "SOC 2 compliant"
#
# ---------------------------------------------------------------------------
# WHAT IS FINE-TUNING? (deep but easy)
# ---------------------------------------------------------------------------
# Fine-tuning = take a model that ALREADY learned a general skill, then keep
# training it a bit more on YOUR smaller, specialized dataset so it gets
# better at your job.
#
# Picture:
#   1) PRETRAIN (already done by OpenAI for Whisper)
#        Millions of hours of speech → model learns English sounds & spelling.
#        Like finishing school.
#
#   2) FINE-TUNE (what YOU do in this notebook)
#        Show pairs:  [sales audio] → "The Professional plan costs $299…"
#        Update weights (or LoRA adapters) so mistakes on sales words shrink.
#        Like an internship: same brain, new workplace vocabulary.
#
#   3) INFERENCE (your live agent)
#        Use the tuned model to transcribe new calls. No labels needed then.
#
# What changes under the hood?
#   Training still minimizes a loss (Whisper: predict next transcript tokens
#   given the audio). Difference vs "training from scratch":
#     - start from strong pretrained weights (not random)
#     - fewer steps / smaller LR usually
#     - dataset is domain-specific and much smaller
#
# Fine-tuning vs related ideas:
#   Transfer learning  — broad name for "reuse pretrained knowledge"
#   Fine-tuning        — the common transfer method: continue gradient updates
#                        on the pretrained net (full or LoRA)
#   Prompting / RAG    — change INPUTS, not weights (no training)
#   SFT (week2)        — fine-tuning a text LLM on instruction→answer flashcards
#                        (same IDEA as here; different modality: text vs speech)
#
# Sticky one-liner:
#   Fine-tuning = specialized practice for a pretrained model.
#
# ---------------------------------------------------------------------------
# FULL FINE-TUNE vs LoRA vs QLoRA (slow walkthrough)
# ---------------------------------------------------------------------------
# Think of Whisper as a HUGE binder of knobs (millions of weights).
# Fine-tuning means: turn some knobs so sales audio is recognized better.
#
# --- 1) FULL FINE-TUNE (Full FT) ---
#   What: unlock EVERY knob and train them all on your sales clips.
#   Pros: maximum flexibility; can change behavior a lot.
#   Cons:
#     - Needs a strong GPU / lots of memory
#     - With only 50–100 clips, the model can MEMORIZE those clips
#       and forget general English (catastrophic forgetting / overfit)
#     - Checkpoint is HUGE (you save the whole model again)
#
#   Picture: repainting the entire house to match one room's style.
#
# --- 2) LoRA (Low-Rank Adaptation) ---
#   What: FREEZE the original Whisper knobs (leave school knowledge intact).
#         Add tiny extra matrices (adapters) next to certain layers
#         (often attention projections). Train ONLY those tiny adapters.
#
#   Math intuition (no pain):
#     A big weight update would be a giant matrix ΔW.
#     LoRA says: approximate ΔW ≈ A × B where A,B are skinny/small ("low rank").
#     Far fewer numbers to learn → cheaper + less overfit on tiny data.
#
#   Pros:
#     - Train ~1% (or less) of parameters
#     - Small adapter file (MBs), base Whisper stays shared
#     - Swap adapters: sales_lora.pt vs medical_lora.pt without copying base
#   Cons:
#     - Slightly less flexible than full FT if you need a huge behavior change
#
#   Picture: leave the house painted; stick removable STYLE DECALS on doors.
#            Want a new domain? Peel decals, stick different ones.
#
#       ┌─────────────────────────────┐
#       │  Frozen Whisper weights     │  ← not updated
#       │    + LoRA adapters (train)  │  ← only these learn sales words
#       └─────────────────────────────┘
#
# --- 3) QLoRA (Quantized LoRA) ---
#   What: same LoRA idea, but the FROZEN base is stored in 4-bit (compressed
#         numbers) to save GPU RAM. Adapters still train in higher precision.
#
#   Quantization (simple):
#     Store weights with fewer bits (less detail) ≈ zip file for numbers.
#     Model is a bit "blurrier" but much smaller in memory.
#
#   Pros: fine-tune bigger Whisper variants on smaller GPUs / laptops
#   Cons: setup is pickier (bitsandbytes, GPU drivers); tiny quality tradeoff
#
#   Picture: keep the house as a compressed photo album (4-bit) + train
#            the same small decals (LoRA) on top.
#
# --- Which should YOU use for this sales Whisper project? ---
#   Tiny demo dataset (tens of clips) → prefer LoRA (or QLoRA if VRAM tight)
#   Full FT → only if you have lots of labeled audio + serious GPU
#
# Sticky cheat sheet:
#   Full FT = retrain whole brain
#   LoRA    = freeze brain, train small stickers
#   QLoRA   = freeze a COMPRESSED brain, train small stickers
#
# ---------------------------------------------------------------------------
# ABBREVIATIONS
# ---------------------------------------------------------------------------
#   ASR  = Automatic Speech Recognition — speech → text (Whisper)
#   WER  = Word Error Rate — % of words wrong vs a gold transcript
#          (insertions + deletions + substitutions) / N_ref_words
#   LoRA = Low-Rank Adaptation — parameter-efficient fine-tuning method
#   TTS  = Text-To-Speech — can SYNTHESIZE training audio from scripts
#          (good for bootstrapping; real calls are better later)
#
# ---------------------------------------------------------------------------
# DATA YOU NEED
# ---------------------------------------------------------------------------
#   Pairs:  audio.wav  +  exact text that was said
#   Start:  50–100 sales lines (TTS or read aloud)
#   Better: real anonymized call snippets with human transcripts
#
# Pipeline in this notebook:
#   1) Build (audio, text) dataset
#   2) Load whisper-tiny.en + processor
#   3) (Later cells) LoRA train → evaluate WER → plug into live agent ASR
#
print("Whisper domain fine-tuning map loaded.")


Whisper domain fine-tuning map loaded.


In [1]:
# =============================================================================
# PREPARE SALES AUDIO + TRANSCRIPTS — (audio path, text) pairs for Whisper
# =============================================================================
#
# Goal of this cell:
#   Build a HuggingFace Dataset with columns like:
#     audio  → waveform @ 16 kHz (what Whisper hears)
#     text   → gold transcript (what Whisper should type)
#
# Production: real calls. Here: create a tiny sales_audio/ folder automatically
# if you don't have one yet (placeholder tones + real sales sentences).
# Replace with TTS (pyttsx3/Coqui) or mic recordings when you go serious.
#
# Terms:
#   WhisperProcessor — turns audio↔features and text↔token ids for Whisper
#   WhisperForConditionalGeneration — the seq2seq ASR model
#   sampling_rate 16000 — Whisper English checkpoints expect 16 kHz mono
#   metadata.csv — simple table: file name + transcript
#

import os
from pathlib import Path
import json
import numpy as np
import pandas as pd
import wave

from datasets import Dataset, Audio
from transformers import WhisperProcessor, WhisperForConditionalGeneration

AUDIO_DIR = Path("sales_audio")
META_CSV = AUDIO_DIR / "metadata.csv"
SAMPLE_RATE = 16000

# Domain lines you CARE about (pricing, compliance, plan names)
SALES_LINES = [
    ("call1.wav", "The Professional plan costs $299 per month."),
    ("call2.wav", "We are SOC 2 Type II compliant."),
    ("call3.wav", "Starter is $99 and includes 24/7 support."),
    ("call4.wav", "Enterprise customers get phone support with a one hour SLA."),
    ("call5.wav", "Rate limits are 5000 requests per minute for Professional."),
]


def _write_placeholder_wav(path: Path, seconds: float = 1.0, freq: float = 220.0):
    """Write a short mono 16-bit WAV (tone). Structure demo only — not real speech."""
    t = np.linspace(0, seconds, int(SAMPLE_RATE * seconds), endpoint=False)
    # Quiet tone so the file is valid audio; swap for TTS speech later
    audio = (0.1 * np.sin(2 * np.pi * freq * t) * 32767).astype(np.int16)
    with wave.open(str(path), "wb") as wf:
        wf.setnchannels(1)
        wf.setsampwidth(2)
        wf.setframerate(SAMPLE_RATE)
        wf.writeframes(audio.tobytes())


# Create folder + CSV + wavs if missing
AUDIO_DIR.mkdir(parents=True, exist_ok=True)
if not META_CSV.exists():
    rows = []
    for i, (fname, text) in enumerate(SALES_LINES):
        wav_path = AUDIO_DIR / fname
        if not wav_path.exists():
            _write_placeholder_wav(wav_path, seconds=1.0 + 0.1 * i, freq=200 + 20 * i)
        rows.append({"file": fname, "text": text})
    pd.DataFrame(rows).to_csv(META_CSV, index=False)
    print(f"Created {META_CSV} with {len(rows)} placeholder clips.")
else:
    print(f"Using existing {META_CSV}")

df = pd.read_csv(META_CSV)
# Prefer paths relative to this notebook's folder layout
df["audio"] = df["file"].apply(lambda f: str(AUDIO_DIR / f))

dataset = Dataset.from_pandas(df)
dataset = dataset.cast_column("audio", Audio(sampling_rate=SAMPLE_RATE))

print("Sample row keys:", dataset[0].keys())
print("text:", dataset[0]["text"])
print("audio sampling_rate:", dataset[0]["audio"]["sampling_rate"])
print("audio array length:", len(dataset[0]["audio"]["array"]))

# Base English Whisper (tiny = fast for learning; upgrade to small/medium later)
processor = WhisperProcessor.from_pretrained("openai/whisper-tiny.en")
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-tiny.en")

print("Loaded openai/whisper-tiny.en | trainable params:",
      sum(p.numel() for p in model.parameters()))
print("Next: freeze base + attach LoRA, then train on these (audio, text) pairs.")
print("NOTE: placeholder WAVs teach the pipeline; for real WER gains use TTS/real speech.")


ModuleNotFoundError: No module named 'datasets'

In [ ]:
# Preprocess Data (Tokenize Audio & Text)

processor = WhisperProcessor.from_pretrained("openai/whisper-tiny.en")

def prepare_dataset(batch):
    # Load and resample audio
    audio = batch["audio"]
    # compute log-Mel spectrogram input features from audio
    batch["input_features"] = processor.feature_extractor(
        audio["array"], sampling_rate=16000
    ).input_features[0]

    # encode target text to label ids
    batch["labels"] = processor.tokenizer(batch["text"]).input_ids
    return batch

# Apply to a few samples (or whole dataset)
dataset = dataset.map(prepare_dataset, remove_columns=dataset.column_names)
dataset.set_format(type="torch", columns=["input_features", "labels"])